"""
==============================================================
CreditCardFraudAI

Notebook 06
Explainable AI using SHAP

Author:
Yogesh Ahuja

Objective
---------
Understand WHY the trained model predicts
fraud by using SHAP explainability.

Outputs
-------
• Global Feature Importance
• SHAP Summary Plot
• Beeswarm Plot
• Dependence Plot
• Waterfall Plot
• Force Plot
• Local Transaction Explanation

==============================================================
"""

In [1]:
# =============================================================================
# Project Setup
# =============================================================================

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: d:\technology\Diploma\upgrad\May-2025-Batch\capstone-project\CreditCardFraudAI


In [2]:
from src.evaluation.model_persistence import ModelPersistence

print(ModelPersistence)

<class 'src.evaluation.model_persistence.ModelPersistence'>


In [3]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import shap

from src.core.config import ConfigManager
from src.explainability.shap_explainer import SHAPExplainer
from src.evaluation.model_persistence import ModelPersistence

In [4]:
import os
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\technology\Diploma\upgrad\May-2025-Batch\capstone-project\CreditCardFraudAI


In [5]:
config = ConfigManager()

print(config)

ConfigManager(config_path='D:\technology\Diploma\upgrad\May-2025-Batch\capstone-project\CreditCardFraudAI\config\config.yaml')


In [6]:
from pathlib import Path

for folder in ["raw", "interim", "processed"]:
    path = PROJECT_ROOT / "data" / folder
    print(f"\n{folder.upper()}")

    if path.exists():
        for file in path.iterdir():
            print("   ", file.name)


RAW
    creditcard.csv

INTERIM

PROCESSED
    X_test.csv
    X_train.csv
    y_test.csv
    y_train.csv


In [7]:
# Load processed datasets

processed_dir = (
    PROJECT_ROOT /
    "data" /
    "processed"
)

X_train = pd.read_csv(processed_dir / "X_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")

y_train = pd.read_csv(
    processed_dir / "y_train.csv"
).squeeze("columns")

y_test = pd.read_csv(
    processed_dir / "y_test.csv"
).squeeze("columns")

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_test.head()

X_train : (453204, 30)
X_test  : (56746, 30)
y_train : (453204,)
y_test  : (56746,)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,-0.705990,0.627767,-0.035995,0.180643,0.459935,-0.036283,0.280205,-0.184115,0.068524,0.586363,...,-0.125656,-0.178453,-0.115609,-0.243481,-1.156797,1.148811,1.019101,0.003099,0.003765,-0.307400
1,1.275941,-0.107325,0.717078,-0.504553,-0.364526,0.455932,-0.540971,0.518948,0.207142,-0.142303,...,-0.128076,0.531835,1.665545,-0.132999,0.839086,-1.363483,-0.486792,0.954855,0.795069,-0.345579
2,-1.346436,-0.861773,0.853484,0.995789,1.540436,0.506870,0.798300,0.906245,-0.453727,-0.229474,...,1.075719,-0.726031,-0.564113,-0.449383,-1.398029,-0.297809,-0.129722,0.015198,0.345560,0.011211
3,-0.876311,0.417583,-0.680621,0.340305,0.365519,-0.734111,0.367605,-0.474457,0.159909,-0.910593,...,-0.143291,-0.078511,-0.231932,-0.318951,-0.557477,0.458099,-0.600772,0.092107,0.176386,0.557220
4,0.729091,1.028625,0.066661,-1.162853,0.418106,0.270858,-0.478062,0.006076,-0.109098,0.451136,...,-0.176194,0.012903,0.493285,-0.021918,-0.738302,0.214409,1.333454,-0.097945,-0.133989,-0.347696


In [8]:
model_path = (
    PROJECT_ROOT /
    "models" /
    "RandomForestClassifier.pkl"
)

model = ModelPersistence.load_model(model_path)

print(type(model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [9]:
# =============================================================================
# Prepare Feature Matrix for Explainability
# =============================================================================
# We use the processed TEST dataset because explainability should be performed
# on unseen data instead of training data.

X = X_test
y = y_test

print("X Shape :", X.shape)
print("y Shape :", y.shape)

X.head()

X Shape : (56746, 30)
y Shape : (56746,)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,-0.705990,0.627767,-0.035995,0.180643,0.459935,-0.036283,0.280205,-0.184115,0.068524,0.586363,...,-0.125656,-0.178453,-0.115609,-0.243481,-1.156797,1.148811,1.019101,0.003099,0.003765,-0.307400
1,1.275941,-0.107325,0.717078,-0.504553,-0.364526,0.455932,-0.540971,0.518948,0.207142,-0.142303,...,-0.128076,0.531835,1.665545,-0.132999,0.839086,-1.363483,-0.486792,0.954855,0.795069,-0.345579
2,-1.346436,-0.861773,0.853484,0.995789,1.540436,0.506870,0.798300,0.906245,-0.453727,-0.229474,...,1.075719,-0.726031,-0.564113,-0.449383,-1.398029,-0.297809,-0.129722,0.015198,0.345560,0.011211
3,-0.876311,0.417583,-0.680621,0.340305,0.365519,-0.734111,0.367605,-0.474457,0.159909,-0.910593,...,-0.143291,-0.078511,-0.231932,-0.318951,-0.557477,0.458099,-0.600772,0.092107,0.176386,0.557220
4,0.729091,1.028625,0.066661,-1.162853,0.418106,0.270858,-0.478062,0.006076,-0.109098,0.451136,...,-0.176194,0.012903,0.493285,-0.021918,-0.738302,0.214409,1.333454,-0.097945,-0.133989,-0.347696


In [10]:
# =============================================================================
# Initialize SHAP Explainer
# =============================================================================
# Background data is sampled from the training dataset.
# SHAP uses this as a reference distribution.

explainer = SHAPExplainer(

    model=model,

    feature_names=X.columns.tolist(),

    output_dir=PROJECT_ROOT / "artifacts" / "explainability",

    background_data=X_train.sample(
        min(1000, len(X_train)),
        random_state=42,
    ),
)

print("SHAP Explainer initialized successfully.")

2026-07-19 00:56:54 | INFO     | SHAPExplainer | Initializing SHAP Explainer...
2026-07-19 00:56:54 | INFO     | SHAPExplainer | Background dataset shape : (1000, 30)
2026-07-19 00:56:54 | INFO     | SHAPExplainer | Selecting SHAP explainer...
2026-07-19 00:56:54 | INFO     | SHAPExplainer | TreeExplainer selected.


SHAP Explainer initialized successfully.


In [11]:
# =============================================================================
# Compute SHAP Values
# =============================================================================

sample_data = X.sample(
    min(500, len(X)),
    random_state=42,
)

explainer.compute_shap_values(sample_data)

print("SHAP values computed successfully.")

2026-07-19 00:56:54 | INFO     | SHAPExplainer | Computing SHAP values for 500 records...
2026-07-19 00:58:21 | INFO     | SHAPExplainer | SHAP values computed successfully.


SHAP values computed successfully.


In [12]:
# =============================================================================
# Global Feature Importance
# =============================================================================

importance = explainer.get_feature_importance()

importance.head(20)

,Feature,Importance
0,V14,0.078541
1,V12,0.078529
2,V4,0.061190
3,V3,0.052716
4,V10,0.047319
5,V11,0.040179
6,V17,0.030431
7,V16,0.019742
8,V1,0.009378
9,V2,0.009155


In [13]:
explainer.summary_plot(
    save_path=PROJECT_ROOT /
    "artifacts/explainability/summary.png"
)

2026-07-19 00:58:21 | INFO     | SHAPExplainer | Generating summary plot...
2026-07-19 00:58:21 | INFO     | SHAPExplainer | Summary plot saved to d:\technology\Diploma\upgrad\May-2025-Batch\capstone-project\CreditCardFraudAI\artifacts\explainability\summary.png


In [14]:
# =============================================================================
# SHAP Beeswarm Plot
# =============================================================================

try:
    explainer.beeswarm_plot(
        save_path=PROJECT_ROOT /
        "artifacts" /
        "explainability" /
        "beeswarm.png"
    )

    print("✓ Beeswarm plot generated.")

except Exception as ex:
    print(f"Skipping Beeswarm Plot: {ex}")

2026-07-19 00:58:21 | INFO     | SHAPExplainer | Generating beeswarm plot...


✓ Beeswarm plot generated.


In [15]:
# =============================================================================
# SHAP Beeswarm Plot
# =============================================================================

explainer.beeswarm_plot(

    save_path=PROJECT_ROOT /
    "artifacts" /
    "explainability" /
    "beeswarm.png",

    show=True,
)

2026-07-19 00:58:22 | INFO     | SHAPExplainer | Generating beeswarm plot...


In [16]:
explainer.dependence_plot(
    feature_name="V14",

    save_path=PROJECT_ROOT /
    "artifacts/explainability/v14_dependence.png"
)

In [17]:
# =============================================================================
# SHAP Bar Plot
# =============================================================================

explainer.bar_plot(

   

    save_path=PROJECT_ROOT /
    "artifacts" /
    "explainability" /
    "bar.png",

    show=True,
)

In [18]:
# =============================================================================
# SHAP Waterfall Plot
# =============================================================================

explainer.waterfall_plot(

    row_index=0,

    save_path=PROJECT_ROOT /
    "artifacts" /
    "explainability" /
    "waterfall.png",

    show=True,
)

2026-07-19 00:58:23 | INFO     | SHAPExplainer | Generating waterfall plot...


In [19]:
# =============================================================================
# SHAP Force Plot
# =============================================================================

explainer.force_plot(

    row_index=0,

    save_path=PROJECT_ROOT /
    "artifacts" /
    "explainability" /
    "force.html",
)

2026-07-19 00:58:23 | INFO     | SHAPExplainer | Generating SHAP force plot...
2026-07-19 00:58:24 | INFO     | SHAPExplainer | Force plot saved to d:\technology\Diploma\upgrad\May-2025-Batch\capstone-project\CreditCardFraudAI\artifacts\explainability\force.html


In [20]:
# =============================================================================
# Explain a Single Fraud Prediction
# =============================================================================

report = explainer.explain_prediction(
    row_index=0,
)

report["feature_contributions"].head(10)

,Feature,Value,SHAP,ABS_SHAP
14,V14,0.300686,-0.093100,0.093100
12,V12,0.036745,-0.083525,0.083525
4,V4,-0.277322,-0.057399,0.057399
10,V10,-0.072736,-0.046462,0.046462
11,V11,-0.121390,-0.036902,0.036902
17,V17,-1.152458,-0.031538,0.031538
16,V16,0.868995,-0.024438,0.024438
8,V8,0.712007,-0.017033,0.017033
24,V24,1.719743,-0.013172,0.013172
6,V6,2.634647,-0.012475,0.012475


In [21]:
# =============================================================================
# Top Positive Contributors
# =============================================================================

explainer.top_positive_features(
    row_index=0
)


,Feature,Value,Contribution
0,V28,0.123124,-0.001088
1,Amount,0.302467,-0.001126
2,V25,0.841508,-0.001798
3,V23,-0.286471,-0.001856
4,V18,0.422762,-0.001871
5,V22,-0.451039,-0.002313
6,V27,-0.114691,-0.002385
7,V13,0.130916,-0.002842
8,Time,-1.229758,-0.002875
9,V20,0.460319,-0.003179


In [22]:
# =============================================================================
# Top Negative Contributors
# =============================================================================

explainer.top_negative_features(
    row_index=0
)

,Feature,Value,Contribution
0,V14,0.300686,-0.093100
1,V12,0.036745,-0.083525
2,V4,-0.277322,-0.057399
3,V10,-0.072736,-0.046462
4,V11,-0.121390,-0.036902
5,V17,-1.152458,-0.031538
6,V16,0.868995,-0.024438
7,V8,0.712007,-0.017033
8,V24,1.719743,-0.013172
9,V6,2.634647,-0.012475


In [23]:
print("=" * 60)
print("Explainable AI Completed")
print("=" * 60)

print(f"Model               : {explainer.model_name}")
print(f"Explainer           : {explainer.explainer_name}")
print(f"Features            : {len(X.columns)}")
print(f"Records Explained   : {len(sample_data)}")
print(f"Output Directory    : {PROJECT_ROOT / 'artifacts' / 'explainability'}")
print("=" * 60)

Explainable AI Completed
Model               : RandomForestClassifier
Explainer           : TreeExplainer
Features            : 30
Records Explained   : 500
Output Directory    : d:\technology\Diploma\upgrad\May-2025-Batch\capstone-project\CreditCardFraudAI\artifacts\explainability
